# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [2]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [3]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

/home/vscode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 46.9MiB/s]


[('calories_per_100g', 89.0), ('food_category', 'fruits'), ('food_item', 'banana'), ('keywords', 'banana_fruits'), ('kj_per_100g', 374.0), ('serving_info', '100g')]
Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


[('calories_per_100g', 50.0), ('food_category', '(fruit)juices'), ('food_item', 'banana juice'), ('keywords', 'banana_juice_(fruit)juices'), ('kj_per_100g', 210.0), ('serving_info', '100ml')]
Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [4]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [5]:
# calorie_lookup_tool('bananas')

In [6]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    """,
    tools=[calorie_lookup_tool],
)

In [7]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result.final_output)

- Total calories (typical medium banana + medium apple): about 200 kcal
  - Banana (~118 g): ~105 kcal (89 kcal/100 g)
  - Apple (~182 g): ~95 kcal (52 kcal/100 g)
- Calories per 100 g:
  - Banana: 89 kcal/100 g
  - Apple: 52 kcal/100 g


### Assignment 

In [14]:
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

# results = nutrition_qna.query(query_texts=["What is malnutrician?"], n_results=2)
# for i, doc in enumerate(results["documents"][0]):
#     print(sorted(results["metadatas"][0][i].items()))
#     print(doc)
#     print("\n")

@function_tool
def qna_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up A plain-text version of the nutritionist Q&A dataset.


    Args:
        query: The user query about malnutrition.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the textual answer about nutrition related questions.
    """
    #print("Inside qna_tool...")
    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        answer = results["Answer"][0][i]
        # metadata = results["metadatas"][0][i]
        # food_item = metadata["food_item"].title()
        # calories = metadata["calories_per_100g"]
        # category = metadata["food_category"].title()

        formatted_results.append(
            f"{answer}\n\n "
        )

    return "Answers:\n" + "\n".join(formatted_results)

RAG_agent = Agent(
    name="Nutrition Assistant Assignment",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information and answering questions about malnutition.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    If you need to look up answers for malnutrition related questions, use the qna_tool.
    """,
    tools=[calorie_lookup_tool, qna_tool],
)

with trace("Assignment Exercise for Nutrition Assistant with RAG"):
    result = await Runner.run(
        RAG_agent,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result.final_output)

print("*" * 100)

with trace("Assignment Exercise for Nutrition Assistant with RAG"):
    result = await Runner.run(
        RAG_agent,
        "What are the symptoms of malnutrition?",
    )
    print(result.final_output)

- Bananas: 89 kcal per 100 g
- Apples: 52 kcal per 100 g

Estimated total for typical sizes:
- Banana (~118 g) + Apple (~182 g) ≈ 200 kcal total

If you have exact weights, I can recalc precisely.
****************************************************************************************************
Common symptoms of malnutrition:

- Unintentional weight loss or fat loss; reduced muscle mass
- Fatigue, weakness, feeling cold
- Poor appetite or no appetite
- Edema (swelling, often in ankles/feet or abdomen)
- Dry, scaly skin; hair loss; brittle nails
- Delayed wound healing and frequent infections
- Digestive issues (constipation, bloating)
- Irritability, dizziness, fainting, or confusion
- In children: stunted growth, delayed development, behavioral changes

If you or someone is at risk, seek medical assessment promptly.


### Instructor's Example
@function_tool
def nutrtition_qna_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function too ask a question about nutrition.

    Args:
        query: The question to ask
        max_results: The maximum number of results to return.

    Returns:
        A string containing the question and the answer related to the query.
    """

    results = nuntrition_qna_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        formatted_results.append(doc)

    return "Related answers to your question:\n" + "\n".join(formatted_results)
    